# LeetCode 257 — Binary Tree Paths bằng Python

- Link đề: https://leetcode.com/problems/binary-tree-paths/
- Độ khó: **Easy** · Chủ đề: Tree, Depth-First Search, Backtracking, String

Chữ ký hàm LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def binaryTreePaths(self, root: TreeNode | None) -> list[str]:

```

**Mục tiêu học**

1. Hiểu cây nhị phân, nút **lá (leaf)**, và khái niệm **đường đi từ gốc tới lá**.
2. Tự dựng `TreeNode` + hàm `build_tree` từ mảng kiểu LeetCode `[1,2,3,null,5]` để chạy độc lập.
3. Giải bằng **DFS đệ quy** mang theo chuỗi đường đi; hiểu call stack qua trace.
4. Học **backtracking**: dùng một list `path` chung, thêm khi đi xuống, bỏ khi quay lên.
5. Viết bản **lặp bằng stack** và **BFS bằng queue**, so sánh độ phức tạp.

## 1. Đề bài và ví dụ

Cho `root` của một cây nhị phân. Trả về **mọi đường đi từ gốc tới lá**, theo **thứ tự bất kỳ**. Lá là nút **không có con**.

Mỗi đường đi được in dạng `"1->2->5"`.

Ví dụ 1: `root = [1,2,3,null,5]`

```text
      1
     / \
    2   3
     \
      5
```

Output: `["1->2->5", "1->3"]`

Ví dụ 2: `root = [1]` → `["1"]`

**Constraints:** số nút trong `[1, 100]`; `-100 <= Node.val <= 100`.

Chú ý: giá trị có thể **âm**, ví dụ `"-1->-2"`.

## 2. Nền tảng: `TreeNode` và dựng cây từ mảng LeetCode

LeetCode biểu diễn cây theo **thứ tự từng tầng (level order)**, `null` cho vị trí trống. Ta viết hàm `build_tree` dùng hàng đợi để gán con trái/phải lần lượt.

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(values: list) -> TreeNode | None:
    """Dựng cây từ list level-order, None = không có nút."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

root = build_tree([1, 2, 3, None, 5])
print(root, root.left, root.right, root.left.left, root.left.right)

TreeNode(1) TreeNode(2) TreeNode(3) None TreeNode(5)


`deque` là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu, đều `O(1)`.

Hàm in cây dạng "nằm ngang" giúp nhìn cấu trúc (cây xoay 90°, gốc ở bên trái, con phải ở trên):

In [2]:
def print_tree(node, indent=0):
    if node is None:
        return
    print_tree(node.right, indent + 4)
    print(" " * indent + str(node.val))
    print_tree(node.left, indent + 4)

print_tree(build_tree([1, 2, 3, None, 5]))

    3
1
        5
    2


### Nút lá là gì?

Nút lá: `node.left is None and node.right is None`. Ở ví dụ 1: lá là `5` và `3`. Nút `2` **không** phải lá (có con phải `5`).

In [3]:
def is_leaf(node: TreeNode) -> bool:
    return node.left is None and node.right is None

r = build_tree([1, 2, 3, None, 5])
for node in [r, r.left, r.right, r.left.right]:
    print(node.val, "là lá?", is_leaf(node))

1 là lá? False
2 là lá? False
3 là lá? True
5 là lá? True


## 3. Nền tảng: đệ quy trên cây

Mọi bài cây đều dựa trên một ý: **giải cho nút hiện tại = kết hợp lời giải của cây con trái và cây con phải**. Ví dụ đếm số nút:

In [4]:
def count_nodes(node):
    if node is None:                 # base case: cây rỗng có 0 nút
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

print(count_nodes(build_tree([1, 2, 3, None, 5])))   # 4

4


Đệ quy cần **điều kiện dừng (base case)** — ở đây là `node is None`. Nếu không có, hàm gọi mãi tới khi `RecursionError`.

## 4. Cách trực giác — DFS mang theo chuỗi đường đi

Ý tưởng: đi từ gốc xuống, mang theo **chuỗi đường đi tới nút hiện tại**.

- Tới nút `node`, nối thêm `str(node.val)` vào chuỗi.
- Nếu `node` là lá → chuỗi là một đáp án, thêm vào kết quả.
- Ngược lại → đi tiếp xuống con trái/phải (nếu có), nối thêm `"->"`.

In [5]:
def paths_recursive_string(root: TreeNode | None) -> list[str]:
    result = []

    def dfs(node, path):
        if node is None:
            return
        path += str(node.val)               # chuỗi mới (chuỗi là bất biến)
        if is_leaf(node):
            result.append(path)
            return
        dfs(node.left, path + "->")
        dfs(node.right, path + "->")

    dfs(root, "")
    return result

print(paths_recursive_string(build_tree([1, 2, 3, None, 5])))
print(paths_recursive_string(build_tree([1])))

['1->2->5', '1->3']
['1']


Vì sao không cần "xóa" gì khi quay lên? Chuỗi Python là **bất biến (immutable)**: `path + "->"` tạo chuỗi **mới**, chuỗi `path` của lời gọi cha không bị đổi. Mỗi lời gọi có bản riêng của mình.

In [6]:
p = "1"
q = p + "->2"
print(p, "|", q)     # p không đổi

1 | 1->2


## 5. Trace call stack

Với `[1,2,3,null,5]`:

| Lời gọi | `node` | `path` sau khi nối | Lá? | Hành động |
|---|---|---|---|---|
| dfs(1, "") | 1 | `"1"` | không | gọi trái, phải với `"1->"` |
| dfs(2, "1->") | 2 | `"1->2"` | không | trái `None` (trả ngay), phải `5` |
| dfs(None, "1->2->") | – | – | – | return |
| dfs(5, "1->2->") | 5 | `"1->2->5"` | **có** | thêm vào kết quả |
| dfs(3, "1->") | 3 | `"1->3"` | **có** | thêm vào kết quả |

In [7]:
def trace_paths(root):
    result = []

    def dfs(node, path, depth):
        pad = "    " * depth
        if node is None:
            print(f"{pad}dfs(None) -> quay lại")
            return
        path += str(node.val)
        if is_leaf(node):
            print(f"{pad}dfs({node.val}): path={path!r} là LÁ -> thêm vào kết quả")
            result.append(path)
            return
        print(f"{pad}dfs({node.val}): path={path!r}, đi xuống trái rồi phải")
        dfs(node.left, path + "->", depth + 1)
        dfs(node.right, path + "->", depth + 1)

    dfs(root, "", 0)
    print("Kết quả:", result)
    return result

trace_paths(build_tree([1, 2, 3, None, 5]))

dfs(1): path='1', đi xuống trái rồi phải
    dfs(2): path='1->2', đi xuống trái rồi phải
        dfs(None) -> quay lại
        dfs(5): path='1->2->5' là LÁ -> thêm vào kết quả
    dfs(3): path='1->3' là LÁ -> thêm vào kết quả
Kết quả: ['1->2->5', '1->3']


['1->2->5', '1->3']

Thụt lề theo `depth` cho thấy **độ sâu call stack**. Thứ tự in là thứ tự DFS **tiền thứ tự (preorder)**: nút → trái → phải.

### Nút thắt nhỏ của cách chuỗi

Mỗi lần `path + "->"` tạo chuỗi mới dài tới `O(h)` (h = chiều cao cây). Với cây "que" (mỗi nút một con) tổng chi phí sao chép có thể lên `O(n²)`. Với `n <= 100` không vấn đề, nhưng ta nên biết cách **backtracking** dùng một list chung.

## 6. Lời giải chính — Backtracking với list `path` dùng chung

- `path` là **list** các giá trị trên đường hiện tại.
- Đi xuống nút: `path.append(node.val)`.
- Tới lá: chuyển `path` thành chuỗi `"->".join(...)`, lưu lại.
- Trước khi **quay lên** khỏi nút: `path.pop()` — trả `path` về trạng thái của nút cha. Đây chính là "backtrack".

In [8]:
def paths_backtracking(root: TreeNode | None) -> list[str]:
    result = []
    path = []

    def dfs(node):
        if node is None:
            return
        path.append(node.val)                     # chọn
        if is_leaf(node):
            result.append("->".join(map(str, path)))
        else:
            dfs(node.left)                        # khám phá
            dfs(node.right)
        path.pop()                                # bỏ chọn (backtrack)

    dfs(root)
    return result

print(paths_backtracking(build_tree([1, 2, 3, None, 5])))
print(paths_backtracking(build_tree([-1, -2, -3])))

['1->2->5', '1->3']
['-1->-2', '-1->-3']


`"->".join(map(str, path))`: `map(str, path)` đổi từng số thành chuỗi (vì `join` chỉ nối chuỗi), rồi nối bằng `"->"`.

In [9]:
print("->".join(map(str, [1, 2, 5])))
try:
    "->".join([1, 2, 5])
except TypeError as e:
    print("TypeError:", e)

1->2->5
TypeError: sequence item 0: expected str instance, int found


### Trace backtracking — nhìn `path` thay đổi

In [10]:
def trace_backtracking(root):
    result, path = [], []

    def dfs(node, depth):
        if node is None:
            return
        pad = "  " * depth
        path.append(node.val)
        print(f"{pad}append {node.val:>2} -> path={path}")
        if is_leaf(node):
            result.append("->".join(map(str, path)))
            print(f"{pad}  LÁ: lưu {result[-1]!r}")
        else:
            dfs(node.left, depth + 1)
            dfs(node.right, depth + 1)
        path.pop()
        print(f"{pad}pop    {node.val:>2} -> path={path}")

    dfs(root, 0)
    return result

trace_backtracking(build_tree([1, 2, 3, 4, 5]))

append  1 -> path=[1]
  append  2 -> path=[1, 2]
    append  4 -> path=[1, 2, 4]
      LÁ: lưu '1->2->4'
    pop     4 -> path=[1, 2]
    append  5 -> path=[1, 2, 5]
      LÁ: lưu '1->2->5'
    pop     5 -> path=[1, 2]
  pop     2 -> path=[1]
  append  3 -> path=[1, 3]
    LÁ: lưu '1->3'
  pop     3 -> path=[1]
pop     1 -> path=[]


['1->2->4', '1->2->5', '1->3']

Mỗi `append` có đúng một `pop` tương ứng → sau khi `dfs(root)` xong, `path` lại rỗng. Đây là **invariant của backtracking**.

## 7. Cách 3 — DFS lặp bằng stack

Không dùng đệ quy: stack chứa các cặp `(node, path_string)`. Đẩy con **phải trước, trái sau** để trái được lấy ra trước (giữ thứ tự giống đệ quy).

In [11]:
def paths_iterative_stack(root):
    if root is None:
        return []
    result = []
    stack = [(root, str(root.val))]
    while stack:
        node, path = stack.pop()
        if is_leaf(node):
            result.append(path)
        if node.right:
            stack.append((node.right, path + "->" + str(node.right.val)))
        if node.left:
            stack.append((node.left, path + "->" + str(node.left.val)))
    return result

print(paths_iterative_stack(build_tree([1, 2, 3, None, 5])))

['1->2->5', '1->3']


`if node.right:` — đối tượng `TreeNode` được coi là `True`, `None` là `False`, nên đây là cách viết gọn của `if node.right is not None`.

## 8. Cách 4 — BFS bằng queue

Duyệt theo tầng; đường đi tới lá **nông hơn** sẽ được tìm trước.

In [12]:
def paths_bfs(root):
    if root is None:
        return []
    result = []
    queue = deque([(root, str(root.val))])
    while queue:
        node, path = queue.popleft()
        if is_leaf(node):
            result.append(path)
            continue
        for child in (node.left, node.right):
            if child:
                queue.append((child, f"{path}->{child.val}"))
    return result

print(paths_bfs(build_tree([1, 2, 3, None, 5])))   # "1->3" (nông hơn) ra trước

['1->3', '1->2->5']


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def binaryTreePaths(self, root: TreeNode | None) -> list[str]:
        result = []
        path = []

        def dfs(node):
            if node is None:
                return
            path.append(str(node.val))
            if node.left is None and node.right is None:
                result.append("->".join(path))
            else:
                dfs(node.left)
                dfs(node.right)
            path.pop()

        dfs(root)
        return result

sol = Solution()
print(sol.binaryTreePaths(build_tree([1, 2, 3, None, 5])))
print(sol.binaryTreePaths(build_tree([1])))

['1->2->5', '1->3']
['1']


Ở bản nộp, `path` lưu sẵn **chuỗi** (`str(node.val)`) nên `"->".join(path)` không cần `map`.

Bản ngắn dùng chuỗi (cũng Accepted):

```python
class Solution:
    def binaryTreePaths(self, root):
        res = []
        def dfs(node, path):
            if not node:
                return
            path += str(node.val)
            if not node.left and not node.right:
                res.append(path)
            else:
                dfs(node.left, path + "->")
                dfs(node.right, path + "->")
        dfs(root, "")
        return res
```

## 10. Vì sao đúng?

**Invariant:** khi `dfs(node)` bắt đầu (sau `append`), `path` chứa đúng các giá trị trên đường từ gốc tới `node`, theo thứ tự.

- Gốc: `path = [root.val]` ✓.
- Nếu đúng cho `node`, khi gọi `dfs(node.left)` thì `append(node.left.val)` → đúng cho con. Sau khi con trả về, `pop` đưa `path` về đúng trạng thái của `node` trước khi gọi con phải.
- Mỗi lá được thăm **đúng một lần** (DFS thăm mỗi nút một lần) và tại lá ta lưu đúng đường gốc→lá. Mỗi đường gốc→lá kết thúc ở một lá duy nhất ⇒ kết quả gồm **đúng tất cả** các đường, không trùng.
- Chỉ lưu tại lá (cả hai con `None`), không lưu tại nút có một con → không sinh đường "dừng giữa chừng".

## 11. Độ phức tạp

`n` = số nút, `h` = chiều cao cây, `L` = số lá.

| Cách | Time | Space phụ (không tính output) |
|---|---|---|
| DFS chuỗi | `O(n · h)` (sao chép chuỗi mỗi cạnh) | `O(h²)` chuỗi trên stack |
| **Backtracking list** | `O(n + L · h)` (join tại mỗi lá) | `O(h)` (`path` + call stack) |
| Stack lặp | `O(n · h)` | `O(n · h)` tệ nhất |
| BFS | `O(n · h)` | `O(n · h)` tệ nhất |

Output có `L` chuỗi, mỗi chuỗi độ dài `O(h)` → kích thước output `O(L · h)`, đây cũng là cận dưới. Cây cân bằng `h = O(log n)`; cây "que" `h = O(n)` (nhưng khi đó chỉ có 1 lá).

## 12. Bộ test

Output theo thứ tự bất kỳ → so sánh sau khi `sorted`.

In [14]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"PASS  {name}: {sorted(actual)}")

sol = Solution()
check("ví dụ 1", sol.binaryTreePaths(build_tree([1, 2, 3, None, 5])), ["1->2->5", "1->3"])
check("ví dụ 2", sol.binaryTreePaths(build_tree([1])), ["1"])
check("giá trị âm", sol.binaryTreePaths(build_tree([-1, -2, 3])), ["-1->-2", "-1->3"])
check("cây đầy đủ", sol.binaryTreePaths(build_tree([1, 2, 3, 4, 5, 6, 7])),
      ["1->2->4", "1->2->5", "1->3->6", "1->3->7"])
check("chỉ con trái (que)", sol.binaryTreePaths(build_tree([1, 2, None, 3, None, 4])), ["1->2->3->4"])
check("nút một con không phải lá", sol.binaryTreePaths(build_tree([1, None, 2])), ["1->2"])
check("giá trị trùng", sol.binaryTreePaths(build_tree([0, 0, 0])), ["0->0", "0->0"])
check("cây rỗng (ngoài constraints)", sol.binaryTreePaths(None), [])

PASS  ví dụ 1: ['1->2->5', '1->3']
PASS  ví dụ 2: ['1']
PASS  giá trị âm: ['-1->-2', '-1->3']
PASS  cây đầy đủ: ['1->2->4', '1->2->5', '1->3->6', '1->3->7']
PASS  chỉ con trái (que): ['1->2->3->4']
PASS  nút một con không phải lá: ['1->2']
PASS  giá trị trùng: ['0->0', '0->0']
PASS  cây rỗng (ngoài constraints): []


In [15]:
import random
rng = random.Random(257)

def random_tree(n_nodes):
    """Tạo cây ngẫu nhiên n_nodes nút bằng cách gắn dần vào chỗ trống."""
    if n_nodes == 0:
        return None
    root = TreeNode(rng.randint(-100, 100))
    nodes = [root]
    while len(nodes) < n_nodes:
        parent = rng.choice(nodes)
        side = rng.choice(["left", "right"])
        if getattr(parent, side) is None:
            child = TreeNode(rng.randint(-100, 100))
            setattr(parent, side, child)
            nodes.append(child)
    return root

def count_leaves(node):
    if node is None:
        return 0
    if is_leaf(node):
        return 1
    return count_leaves(node.left) + count_leaves(node.right)

solvers = [paths_recursive_string, paths_backtracking, paths_iterative_stack, paths_bfs,
           Solution().binaryTreePaths]
for _ in range(500):
    t = random_tree(rng.randint(1, 100))
    outs = [sorted(f(t)) for f in solvers]
    assert all(o == outs[0] for o in outs)
    assert len(outs[0]) == count_leaves(t)
print("500 cây ngẫu nhiên (1..100 nút): 5 cài đặt khớp nhau, số đường = số lá — PASS")

500 cây ngẫu nhiên (1..100 nút): 5 cài đặt khớp nhau, số đường = số lá — PASS


`getattr(obj, "left")` đọc thuộc tính theo tên chuỗi; `setattr(obj, "left", value)` gán thuộc tính theo tên — tiện để chọn ngẫu nhiên trái/phải.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Lưu đường đi khi gặp `None` thay vì khi gặp lá

In [16]:
def wrong_save_at_none(root):
    res = []
    def dfs(node, path):
        if node is None:
            res.append(path[:-2])          # cắt "->" thừa
            return
        dfs(node.left, path + str(node.val) + "->")
        dfs(node.right, path + str(node.val) + "->")
    dfs(root, "")
    return res

print("sai :", wrong_save_at_none(build_tree([1, 2, 3, None, 5])))

sai : ['1->2', '1->2->5', '1->2->5', '1->3', '1->3']


**Vì sao sai:** nút `2` có con trái `None` → lưu `"1->2"` dù `2` không phải lá; mỗi lá cũng bị lưu **hai lần** (một cho mỗi con `None`). **Sửa:** chỉ lưu khi `node.left is None and node.right is None`.

### Lỗi 2: Quên `path.pop()` trong backtracking

In [17]:
def wrong_no_pop(root):
    res, path = [], []
    def dfs(node):
        if node is None:
            return
        path.append(str(node.val))
        if not node.left and not node.right:
            res.append("->".join(path))
        dfs(node.left)
        dfs(node.right)
        # thiếu path.pop()
    dfs(root)
    return res

print("sai :", wrong_no_pop(build_tree([1, 2, 3, None, 5])))

sai : ['1->2->5', '1->2->5->3']


**Vì sao sai:** sau khi xong nhánh `2 -> 5`, `path` vẫn còn `[1, 2, 5]`, nên khi sang `3` thành `[1, 2, 5, 3]`. **Sửa:** `pop` mỗi lần rời một nút.

### Lỗi 3: Lưu chính list `path` (tham chiếu) thay vì bản sao

In [18]:
def wrong_save_reference(root):
    res, path = [], []
    def dfs(node):
        if node is None:
            return
        path.append(node.val)
        if not node.left and not node.right:
            res.append(path)            # lưu tham chiếu tới CÙNG một list
        dfs(node.left); dfs(node.right)
        path.pop()
    dfs(root)
    return res

print("sai :", wrong_save_reference(build_tree([1, 2, 3, None, 5])))

sai : [[], []]


**Vì sao sai:** mọi phần tử của `res` trỏ tới **cùng một** list `path`, mà cuối cùng `path` rỗng → `[[], []]`. **Sửa:** lưu bản sao `path[:]` / `list(path)` hoặc chuỗi `"->".join(...)` (chuỗi mới, độc lập).

### Lỗi 4: Quên đổi số sang chuỗi
`"->".join([1, 2, 5])` → `TypeError` (đã thấy ở mục 6). Hoặc `path + node.val` → `TypeError: can only concatenate str (not "int") to str`. **Sửa:** `str(node.val)`.

### Lỗi 5: In thừa `"->"` ở cuối
Nối `"->"` **sau** mỗi giá trị rồi lưu ở lá sẽ ra `"1->3->"`. Cách an toàn nhất là `join` (chỉ chèn giữa các phần tử).

## 14. Mẹo nhận dạng pattern

- "Liệt kê **mọi** đường đi / mọi tổ hợp" → **DFS + backtracking** (append → đệ quy → pop).
- "Gốc tới lá" → điều kiện lá `not node.left and not node.right`, xử lý **tại lá**.
- Dữ liệu đường đi bất biến (chuỗi, tuple) → truyền theo tham số, không cần pop. Dữ liệu thay đổi được (list) → nhớ pop hoặc sao chép khi lưu.
- Cần đường đi ngắn nhất/nông nhất trước → BFS.

## 15. Biến thể

- **Path Sum II (113):** mọi đường gốc→lá có tổng bằng `target`.
- **Sum Root to Leaf Numbers (129):** mỗi đường là một số, tính tổng.

In [19]:
def path_sum_ii(root, target):
    res, path = [], []
    def dfs(node, remain):
        if node is None:
            return
        path.append(node.val)
        remain -= node.val
        if is_leaf(node) and remain == 0:
            res.append(path[:])              # bản sao!
        else:
            dfs(node.left, remain)
            dfs(node.right, remain)
        path.pop()
    dfs(root, target)
    return res

def sum_numbers(root):
    def dfs(node, cur):
        if node is None:
            return 0
        cur = cur * 10 + node.val
        if is_leaf(node):
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

t = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
print(path_sum_ii(t, 22))
print(sum_numbers(build_tree([1, 2, 3])))   # 12 + 13 = 25

[[5, 4, 11, 2], [5, 8, 4, 5]]
25


## 16. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay
Cây `[1, 2, 3, 4, None, None, 5]`: vẽ cây, liệt kê các lá và mọi đường gốc→lá.

### Bài 2 — Đếm số lá

In [20]:
def exercise_count_leaves(root) -> int:
    # TODO: base case None -> 0; lá -> 1; còn lại cộng hai con
    pass

# assert exercise_count_leaves(build_tree([1, 2, 3, None, 5])) == 2

### Bài 3 — Tự viết `binaryTreePaths` bằng backtracking

In [21]:
def exercise_paths(root) -> list[str]:
    # TODO 1: result, path = [], []
    # TODO 2: dfs: append -> (lá: lưu join) / (không: đệ quy 2 con) -> pop
    pass

# assert sorted(exercise_paths(build_tree([1, 2, 3, None, 5]))) == ["1->2->5", "1->3"]

### Bài 4 — Đường đi dài nhất (nhiều nút nhất) từ gốc tới lá
Trả về chuỗi đường đi dài nhất; nếu hòa, lấy đường **xuất hiện trước** theo DFS trái→phải.

In [22]:
def exercise_longest_path(root) -> str:
    # TODO: dùng kết quả binaryTreePaths theo thứ tự DFS, chọn chuỗi có nhiều "->" nhất
    pass

# assert exercise_longest_path(build_tree([1, 2, 3, None, 5])) == "1->2->5"

### Bài 5 — Tổng các số gốc→lá (LeetCode 129)
Mỗi đường `1->2->5` là số `125`. Tính tổng mọi số. Giá trị nút trong bài này là `0..9`.

In [23]:
def exercise_sum_numbers(root) -> int:
    # TODO: truyền cur = cur * 10 + val xuống con
    pass

# assert exercise_sum_numbers(build_tree([1, 2, 3])) == 25
# assert exercise_sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026

### Bài 6 — Mini challenge: đường đi dạng list, viết **không đệ quy**
Trả về list các list giá trị (không phải chuỗi), dùng stack, mỗi phần tử stack là `(node, path_list)`. Cẩn thận: mỗi nhánh cần **bản sao** riêng của list.

In [24]:
def exercise_paths_lists_iterative(root) -> list[list[int]]:
    # TODO
    pass

# assert sorted(exercise_paths_lists_iterative(build_tree([1, 2, 3, None, 5]))) == [[1, 2, 5], [1, 3]]

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Dựng theo tầng: 2 trái, 3 phải của 1 | `4` là con trái của `2`; `5` là con phải của `3` | lá: 4, 5; đường `1->2->4`, `1->3->5` |
| 2 | Mục 3 (đếm nút) | điều kiện lá | `return count(l) + count(r)` |
| 3 | Mục 6 | `path.append(str(val))` | đừng quên `path.pop()` |
| 4 | Độ dài = số `"->"` + 1 | `max(paths, key=...)` lấy phần tử đầu khi hòa | `key=lambda p: p.count("->")` |
| 5 | Như DFS chuỗi nhưng mang số | lá trả `cur` | `dfs(l, cur) + dfs(r, cur)` |
| 6 | Mục 7 | `path + [child.val]` tạo list mới | đẩy phải trước trái |

## 18. Đáp án tham khảo, template và checklist

Bài 1: cây `1` (trái `2`, phải `3`); `2` có con trái `4`; `3` có con phải `5`. Lá: `4`, `5`. Đường: `"1->2->4"`, `"1->3->5"`.

In [25]:
def answer_count_leaves(root):
    if root is None:
        return 0
    if root.left is None and root.right is None:
        return 1
    return answer_count_leaves(root.left) + answer_count_leaves(root.right)

def answer_paths(root):
    result, path = [], []
    def dfs(node):
        if node is None:
            return
        path.append(str(node.val))
        if node.left is None and node.right is None:
            result.append("->".join(path))
        else:
            dfs(node.left)
            dfs(node.right)
        path.pop()
    dfs(root)
    return result

def answer_longest_path(root):
    return max(answer_paths(root), key=lambda p: p.count("->"))   # max giữ phần tử đầu khi hòa

def answer_sum_numbers(root):
    def dfs(node, cur):
        if node is None:
            return 0
        cur = cur * 10 + node.val
        if node.left is None and node.right is None:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

def answer_paths_lists_iterative(root):
    if root is None:
        return []
    res, stack = [], [(root, [root.val])]
    while stack:
        node, path = stack.pop()
        if node.left is None and node.right is None:
            res.append(path)
        if node.right:
            stack.append((node.right, path + [node.right.val]))
        if node.left:
            stack.append((node.left, path + [node.left.val]))
    return res

print(sorted(answer_paths(build_tree([1, 2, 3, 4, None, None, 5]))))
assert answer_count_leaves(build_tree([1, 2, 3, None, 5])) == 2
assert sorted(answer_paths(build_tree([1, 2, 3, None, 5]))) == ["1->2->5", "1->3"]
assert answer_longest_path(build_tree([1, 2, 3, None, 5])) == "1->2->5"
assert answer_longest_path(build_tree([1, 2, 3])) == "1->2"
assert answer_sum_numbers(build_tree([1, 2, 3])) == 25 and answer_sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026
assert sorted(answer_paths_lists_iterative(build_tree([1, 2, 3, None, 5]))) == [[1, 2, 5], [1, 3]]
for _ in range(200):
    t = random_tree(rng.randint(1, 40))
    assert sorted(answer_paths(t)) == sorted(Solution().binaryTreePaths(t))
    assert answer_count_leaves(t) == count_leaves(t)
    assert sorted("->".join(map(str, p)) for p in answer_paths_lists_iterative(t)) == sorted(answer_paths(t))
print("Đáp án tham khảo: tất cả test PASS")

['1->2->4', '1->3->5']


Đáp án tham khảo: tất cả test PASS


### Template nhớ nhanh (DFS + backtracking)

```python
result, path = [], []
def dfs(node):
    if node is None:
        return
    path.append(str(node.val))            # chọn
    if not node.left and not node.right:  # lá
        result.append("->".join(path))
    else:
        dfs(node.left)                    # khám phá
        dfs(node.right)
    path.pop()                            # bỏ chọn
dfs(root)
return result
```

### Checklist submit

- [ ] Lưu đường đi **chỉ tại lá** (cả hai con `None`).
- [ ] Mỗi `append` có đúng một `pop`.
- [ ] Lưu chuỗi/bản sao, không lưu tham chiếu tới list `path`.
- [ ] `str(node.val)` — giá trị có thể âm, `"-1->-2"` vẫn đúng.
- [ ] Dùng `"->".join(...)` để không thừa `"->"` ở cuối.
- [ ] Nêu được Time `O(n + L·h)` / Space `O(h)` cho backtracking.